# Phase 1 — Part 2: Exploratory Data Analysis

This notebook documents the exploratory data analysis (EDA) performed on the Healthcare Stroke Prediction dataset as the second deliverable of Phase 1. The objective is to characterise the data, quantify missingness, inspect univariate and bivariate structure, surface potential predictors of stroke, and enumerate findings that will guide the subsequent preprocessing and modelling phases. All figures are persisted under `../figures/` with the `02_` prefix so that they can be referenced directly from the project report. **Author:** Markela  

In [1]:
# Standard scientific Python stack and plotting configuration.
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore', category=FutureWarning)

# Seaborn style with a paper-quality context; high DPI for crisp figures.
sns.set_theme(style='whitegrid', context='paper')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.family'] = 'DejaVu Sans'

# Ensure the shared figures directory exists relative to the notebook location.
FIG_DIR = Path('../figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)
print('Figures will be written to:', FIG_DIR.resolve())

Figures will be written to: C:\Users\donat\Desktop\Markie\figures


In [2]:
# Load the raw stroke dataset. The source file encodes missing BMI as the
# string 'N/A'; we coerce it to float NaN so that downstream summaries treat
# the column as numeric. The identifier column carries no modelling signal
# and is therefore dropped before analysis.
DATA_PATH = Path('../data/healthcare-dataset-stroke-data.csv')
df = pd.read_csv(DATA_PATH)
df['bmi'] = pd.to_numeric(df['bmi'].replace('N/A', np.nan), errors='coerce')
df = df.drop(columns=['id'])
print('Loaded dataset shape:', df.shape)
df.head()

Loaded dataset shape: (5110, 11)


,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


## 2.1 Dataset Overview

The Healthcare Stroke Prediction dataset consists of 5,110 patient records and eleven descriptive attributes (after discarding the `id` column). The attributes comprise three continuous numeric measurements (`age`, `avg_glucose_level`, `bmi`), two binary clinical indicators (`hypertension`, `heart_disease`), five categorical demographic or lifestyle variables (`gender`, `ever_married`, `work_type`, `Residence_type`, `smoking_status`), and the binary target `stroke`. The following cell summarises the schema, provides descriptive statistics for numeric columns, and reports the modal categories for the categorical columns.

In [3]:
# Schema inspection and descriptive summaries.
df.info()
print('\nNumeric descriptive statistics:')
display(df.describe().round(3))
print('\nCategorical descriptive statistics:')
display(df.describe(include='object'))

<class 'pandas.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   gender             5110 non-null   str    
 1   age                5110 non-null   float64
 2   hypertension       5110 non-null   int64  
 3   heart_disease      5110 non-null   int64  
 4   ever_married       5110 non-null   str    
 5   work_type          5110 non-null   str    
 6   Residence_type     5110 non-null   str    
 7   avg_glucose_level  5110 non-null   float64
 8   bmi                4909 non-null   float64
 9   smoking_status     5110 non-null   str    
 10  stroke             5110 non-null   int64  
dtypes: float64(3), int64(3), str(5)
memory usage: 439.3 KB

Numeric descriptive statistics:


,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke
count,5110.000,5110.000,5110.000,5110.000,4909.000,5110.000
mean,43.227,0.097,0.054,106.148,28.893,0.049
std,22.613,0.297,0.226,45.284,7.854,0.215
min,0.080,0.000,0.000,55.120,10.300,0.000
25%,25.000,0.000,0.000,77.245,23.500,0.000
50%,45.000,0.000,0.000,91.885,28.100,0.000
75%,61.000,0.000,0.000,114.090,33.100,0.000
max,82.000,1.000,1.000,271.740,97.600,1.000



Categorical descriptive statistics:


C:\Users\donat\AppData\Local\Temp\ipykernel_20496\1781582823.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  display(df.describe(include='object'))


,gender,ever_married,work_type,Residence_type,smoking_status
count,5110,5110,5110,5110,5110
unique,3,2,5,2,4
top,Female,Yes,Private,Urban,never smoked
freq,2994,3353,2925,2596,1892


## 2.2 Missing Values Analysis

Missingness is examined at the column level. Only `bmi` exhibits true numeric missingness (encoded in the source file as the literal string `N/A`). The `smoking_status` feature additionally contains an explicit `Unknown` category which is not NaN in the pandas sense but functions as a missing-label indicator and is treated as informative in later sections. The bar plot below is saved to `../figures/02_missingness.png` for inclusion in the report.

In [4]:
# Quantify and visualise missingness per column.
missing_count = df.isna().sum()
missing_pct = (missing_count / len(df) * 100).round(3)
missing_report = (
    pd.DataFrame({'missing': missing_count, 'percent': missing_pct})
    .sort_values('missing', ascending=False)
)
display(missing_report)

fig, ax = plt.subplots(figsize=(8, 4.5))
plot_df = missing_report[missing_report['missing'] > 0]
if plot_df.empty:
    plot_df = missing_report
sns.barplot(x=plot_df.index, y=plot_df['percent'], ax=ax, color='#3b7dd8')
ax.set_title('Per-feature missingness')
ax.set_ylabel('Missing (%)')
ax.set_xlabel('Feature')
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f%%', padding=3)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(FIG_DIR / '02_missingness.png')
plt.close()

,missing,percent
bmi,201,3.933
age,0,0.000
gender,0,0.000
hypertension,0,0.000
heart_disease,0,0.000
work_type,0,0.000
ever_married,0,0.000
Residence_type,0,0.000
avg_glucose_level,0,0.000
smoking_status,0,0.000


## 2.3 Univariate Analysis — Numeric Features

We now inspect the marginal distributions of the three continuous predictors. `age` is expected to be broadly uniform across adult deciles with a smaller tail at paediatric values; `avg_glucose_level` is likely right-skewed and may exhibit a secondary mode corresponding to a diabetic subpopulation; `bmi` is typically approximately log-normal with positive skew and a long upper tail. The fourth panel of the figure reports the target class counts for orientation.

In [5]:
# Histograms with KDE overlays for each continuous feature plus the target.
numeric_cols = ['age', 'avg_glucose_level', 'bmi']

print('Distributional moments:')
for col in numeric_cols:
    series = df[col].dropna()
    print(
        f"{col:<18s} skewness={stats.skew(series):+.3f}  "
        f"kurtosis={stats.kurtosis(series):+.3f}  "
        f"mean={series.mean():.2f}  median={series.median():.2f}"
    )

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
panels = [
    ('age', 'Age (years)'),
    ('avg_glucose_level', 'Average glucose level (mg/dL)'),
    ('bmi', 'Body mass index'),
]
for ax, (col, label) in zip(axes.flat[:3], panels):
    sns.histplot(df[col].dropna(), kde=True, ax=ax, color='#3b7dd8', edgecolor='white')
    ax.set_title(f'Distribution of {col}')
    ax.set_xlabel(label)
    ax.set_ylabel('Count')

ax4 = axes.flat[3]
sns.countplot(x='stroke', data=df, ax=ax4, hue='stroke',
              palette=['#3b7dd8', '#d8523b'], legend=False)
ax4.set_title('Target variable (stroke)')
ax4.set_xlabel('Stroke')
ax4.set_ylabel('Count')
for container in ax4.containers:
    ax4.bar_label(container, padding=3)

plt.tight_layout()
plt.savefig(FIG_DIR / '02_numeric_distributions.png')
plt.close()

Distributional moments:
age                skewness=-0.137  kurtosis=-0.991  mean=43.23  median=45.00
avg_glucose_level  skewness=+1.572  kurtosis=+1.678  mean=106.15  median=91.88
bmi                skewness=+1.055  kurtosis=+3.358  mean=28.89  median=28.10


## 2.4 Univariate Analysis — Categorical Features

We next examine the five categorical attributes. The `gender` column is notable for containing a single record labelled `Other`, which is retained here so that the EDA reflects the full raw dataset but will warrant attention during preprocessing. `smoking_status` includes an `Unknown` level that encodes informative missingness. `work_type` has a heavy `Private` majority, and `Residence_type` is nearly balanced between `Urban` and `Rural`.

In [6]:
# Value counts and bar charts for the categorical attributes.
categorical_cols = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']

for col in categorical_cols:
    print(f'\n{col} value counts:')
    print(df[col].value_counts(dropna=False))

ncols = 2
nrows = int(np.ceil(len(categorical_cols) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3.5 * nrows))
axes = axes.flatten()
for ax, col in zip(axes, categorical_cols):
    order = df[col].value_counts().index
    sns.countplot(y=col, data=df, ax=ax, order=order, hue=col,
                  palette='viridis', legend=False)
    ax.set_title(f'Distribution of {col}')
    ax.set_xlabel('Count')
    ax.set_ylabel('')
    for container in ax.containers:
        ax.bar_label(container, padding=3)
for ax in axes[len(categorical_cols):]:
    ax.set_visible(False)

plt.tight_layout()
plt.savefig(FIG_DIR / '02_categorical_distributions.png')
plt.close()


gender value counts:
gender
Female    2994
Male      2115
Other        1
Name: count, dtype: int64

ever_married value counts:
ever_married
Yes    3353
No     1757
Name: count, dtype: int64

work_type value counts:
work_type
Private          2925
Self-employed     819
children          687
Govt_job          657
Never_worked       22
Name: count, dtype: int64

Residence_type value counts:
Residence_type
Urban    2596
Rural    2514
Name: count, dtype: int64

smoking_status value counts:
smoking_status
never smoked       1892
Unknown            1544
formerly smoked     885
smokes              789
Name: count, dtype: int64


## 2.5 Target Variable Distribution

The target `stroke` is strongly imbalanced, with approximately 4.9% positive cases. This imbalance has material implications for model selection and evaluation: accuracy alone will be misleading, and the modelling phase should prioritise threshold-independent metrics (ROC AUC, PR AUC) alongside recall at fixed precision and resampling or class-weighting strategies during training.

In [7]:
# Pie chart and count plot of the binary target with an explicit imbalance ratio.
counts = df['stroke'].value_counts().sort_index()
ratio = counts[0] / max(counts[1], 1)
print(counts)
print(f"Positive rate: {df['stroke'].mean() * 100:.2f}%")
print(f'Imbalance ratio (negative : positive) = {ratio:.1f} : 1')

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].pie(
    counts.values,
    labels=['No stroke', 'Stroke'],
    autopct='%1.2f%%',
    colors=['#3b7dd8', '#d8523b'],
    startangle=90,
    wedgeprops={'edgecolor': 'white'},
)
axes[0].set_title('Target composition')

sns.countplot(x='stroke', data=df, ax=axes[1], hue='stroke',
              palette=['#3b7dd8', '#d8523b'], legend=False)
axes[1].set_title('Target counts')
axes[1].set_xlabel('Stroke')
axes[1].set_ylabel('Count')
for container in axes[1].containers:
    axes[1].bar_label(container, padding=3)

plt.tight_layout()
plt.savefig(FIG_DIR / '02_target_distribution.png')
plt.close()

stroke
0    4861
1     249
Name: count, dtype: int64
Positive rate: 4.87%
Imbalance ratio (negative : positive) = 19.5 : 1


## 2.6 Bivariate Analysis — Numeric Features vs Target

We compare the conditional distributions of each continuous feature across stroke and non-stroke groups using violin plots, and quantify the difference with Mann–Whitney U tests. Mann–Whitney is preferred over the Student t-test because the numeric features are not normally distributed. The rank-biserial correlation is reported as a distribution-free effect size.

In [8]:
# Violin plots and Mann-Whitney U tests for each continuous feature.
def rank_biserial(u_stat, n1, n2):
    if n1 == 0 or n2 == 0:
        return float('nan')
    return 1 - (2 * u_stat) / (n1 * n2)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, col in zip(axes, numeric_cols):
    sns.violinplot(x='stroke', y=col, data=df, ax=ax, hue='stroke',
                   palette=['#3b7dd8', '#d8523b'], inner='quartile', legend=False)
    ax.set_title(f'{col} by stroke status')
    ax.set_xlabel('Stroke')
plt.tight_layout()
plt.savefig(FIG_DIR / '02_numeric_vs_target.png')
plt.close()

print(f"{'feature':<20s}{'U':>14s}{'p-value':>14s}{'rank-biserial':>18s}")
for col in numeric_cols:
    g1 = df.loc[df['stroke'] == 1, col].dropna()
    g0 = df.loc[df['stroke'] == 0, col].dropna()
    u_stat, p_val = stats.mannwhitneyu(g1, g0, alternative='two-sided')
    effect = rank_biserial(u_stat, len(g1), len(g0))
    print(f'{col:<20s}{u_stat:>14.1f}{p_val:>14.3e}{effect:>18.3f}')

feature                          U       p-value     rank-biserial
age                      1010125.5     3.727e-71            -0.669
avg_glucose_level         739150.0     3.640e-09            -0.221
bmi                       569021.5     1.026e-04            -0.159


## 2.7 Bivariate Analysis — Categorical Features vs Target

For each categorical attribute we compute the per-category stroke prevalence, compare distributions with a Pearson chi-squared test of independence, and report the bias-corrected Cramér's V as a standardised effect size. This combination identifies which attributes carry statistically reliable and practically meaningful association with the target.

In [9]:
# Stroke rate bar plots and chi-squared tests with Cramer's V.
def cramers_v(contingency):
    chi2 = stats.chi2_contingency(contingency, correction=False)[0]
    n = contingency.sum()
    if n == 0:
        return float('nan')
    phi2 = chi2 / n
    r, k = contingency.shape
    phi2corr = max(0.0, phi2 - ((k - 1) * (r - 1)) / max(n - 1, 1))
    rcorr = r - ((r - 1) ** 2) / max(n - 1, 1)
    kcorr = k - ((k - 1) ** 2) / max(n - 1, 1)
    denom = min(kcorr - 1, rcorr - 1)
    if denom <= 0:
        return float('nan')
    return float(np.sqrt(phi2corr / denom))

ncols = 2
nrows = int(np.ceil(len(categorical_cols) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 3.8 * nrows))
axes = axes.flatten()
for ax, col in zip(axes, categorical_cols):
    rate = df.groupby(col)['stroke'].mean().sort_values(ascending=False)
    sns.barplot(x=rate.values * 100, y=rate.index, ax=ax,
                hue=rate.index, palette='magma', legend=False)
    ax.set_title(f'Stroke rate by {col}')
    ax.set_xlabel('Stroke rate (%)')
    ax.set_ylabel('')
    for container in ax.containers:
        ax.bar_label(container, fmt='%.2f%%', padding=3)
for ax in axes[len(categorical_cols):]:
    ax.set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / '02_categorical_vs_target.png')
plt.close()

print(f"{'feature':<20s}{'chi2':>12s}{'dof':>6s}{'p-value':>14s}{'CramersV':>12s}")
for col in categorical_cols:
    table = pd.crosstab(df[col], df['stroke'])
    chi2, p_val, dof, _ = stats.chi2_contingency(table.values)
    v = cramers_v(table.values)
    print(f'{col:<20s}{chi2:>12.3f}{dof:>6d}{p_val:>14.3e}{v:>12.3f}')

feature                     chi2   dof       p-value    CramersV
gender                     0.473     2     7.895e-01       0.000
ever_married              58.924     1     1.639e-14       0.107
work_type                 49.164     4     5.398e-10       0.094
Residence_type             1.082     1     2.983e-01       0.007
smoking_status            29.147     3     2.085e-06       0.072


## 2.8 Correlation Analysis

We compute Spearman rank correlations across continuous features, the binary clinical flags (`hypertension`, `heart_disease`), several binarised demographic attributes, and the target. Spearman is preferred over Pearson because it is robust to non-linear monotonic relationships and the marked non-normality of `avg_glucose_level` and `bmi`. The ranking of absolute correlations with `stroke` gives a first, unconditional view of feature importance.

In [10]:
# Spearman correlation heatmap on numeric + binary-encoded features.
frame = df.copy()
frame['gender_bin'] = (frame['gender'] == 'Male').astype(int)
frame['ever_married_bin'] = (frame['ever_married'] == 'Yes').astype(int)
frame['residence_bin'] = (frame['Residence_type'] == 'Urban').astype(int)

corr_cols = [
    'age', 'avg_glucose_level', 'bmi',
    'hypertension', 'heart_disease',
    'gender_bin', 'ever_married_bin', 'residence_bin',
    'stroke',
]
corr = frame[corr_cols].corr(method='spearman')

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='vlag', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Spearman correlations (numeric + binary features)')
plt.tight_layout()
plt.savefig(FIG_DIR / '02_correlation_heatmap.png')
plt.close()

top = corr['stroke'].drop('stroke').abs().sort_values(ascending=False)
print('Top features by |Spearman| with stroke:')
for feature, _ in top.items():
    print(f"  {feature:<22s} {corr['stroke'][feature]:+.3f}")

Top features by |Spearman| with stroke:
  age                    +0.250
  heart_disease          +0.135
  hypertension           +0.128
  ever_married_bin       +0.108
  avg_glucose_level      +0.083
  bmi                    +0.055
  residence_bin          +0.015
  gender_bin             +0.009


## 2.9 Outlier Analysis

Outliers are identified per continuous feature using the classical Tukey interquartile range (IQR) rule, flagging values outside `[Q1 - 1.5 IQR, Q3 + 1.5 IQR]`. We view this as an exploratory tag rather than a filter: clinically extreme glucose or BMI values can be genuine and are often the most informative for stroke risk, so removal decisions are deferred to the preprocessing phase.

In [11]:
# IQR outlier counts and supporting boxplots.
print(f"{'feature':<20s}{'lower':>12s}{'upper':>12s}{'outliers':>12s}{'percent':>10s}")
for col in numeric_cols:
    series = df[col].dropna()
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (series < lower) | (series > upper)
    count = int(mask.sum())
    pct = count / len(series) * 100
    print(f'{col:<20s}{lower:>12.2f}{upper:>12.2f}{count:>12d}{pct:>9.2f}%')

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, col in zip(axes, numeric_cols):
    sns.boxplot(x=df[col].dropna(), ax=ax, color='#3b7dd8', fliersize=3)
    ax.set_title(f'{col} (IQR boxplot)')
    ax.set_xlabel(col)
plt.tight_layout()
plt.savefig(FIG_DIR / '02_outliers.png')
plt.close()

feature                    lower       upper    outliers   percent
age                       -29.00      115.00           0     0.00%
avg_glucose_level          21.98      169.36         627    12.27%
bmi                         9.10       47.50         110     2.24%


## 2.10 Multivariate Insights

Univariate summaries can obscure interactions between features. We therefore examine joint structure in two scatter plots: (i) `age` versus `avg_glucose_level`, which exposes the characteristic diabetic cluster at high glucose, and (ii) `age` versus `bmi`, which shows whether BMI contributes beyond age. Stroke cases are highlighted in red. We also report per-decile stroke prevalence to support the narrative that age is the dominant risk axis.

In [12]:
# Targeted scatter plots coloured by stroke, plus age-decile prevalence.
subset = df.dropna(subset=['bmi']).copy()
subset['stroke_label'] = subset['stroke'].map({0: 'No stroke', 1: 'Stroke'})
palette = {'No stroke': '#3b7dd8', 'Stroke': '#d8523b'}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=subset, x='age', y='avg_glucose_level', hue='stroke_label',
                palette=palette, alpha=0.55, s=25, ax=axes[0])
axes[0].set_title('Age vs. average glucose level')
axes[0].set_xlabel('Age (years)')
axes[0].set_ylabel('Average glucose level (mg/dL)')
axes[0].legend(title='')

sns.scatterplot(data=subset, x='age', y='bmi', hue='stroke_label',
                palette=palette, alpha=0.55, s=25, ax=axes[1])
axes[1].set_title('Age vs. body mass index')
axes[1].set_xlabel('Age (years)')
axes[1].set_ylabel('BMI (kg/m^2)')
axes[1].legend(title='')

plt.tight_layout()
plt.savefig(FIG_DIR / '02_multivariate.png')
plt.close()

subset['age_decile'] = pd.cut(subset['age'], bins=np.arange(0, 101, 10), right=False)
decile_rate = subset.groupby('age_decile', observed=True)['stroke'].mean() * 100
print('Stroke rate by age decile (%):')
print(decile_rate.round(2))

Stroke rate by age decile (%):
age_decile
[0, 10)      0.00
[10, 20)     0.21
[20, 30)     0.00
[30, 40)     0.79
[40, 50)     1.70
[50, 60)     5.09
[60, 70)     6.78
[70, 80)    13.84
[80, 90)    21.43
Name: stroke, dtype: float64


## 2.11 Key EDA Findings

The following observations summarise the empirical evidence gathered above. All numerical claims are grounded in the statistics printed in the preceding cells rather than in any prior belief about the dataset.

1. The dataset contains 5,110 patient records across eleven predictive attributes after dropping the identifier, which is adequate for classical supervised learning but small relative to the severity of class imbalance.
2. The target variable `stroke` is highly imbalanced at approximately 4.9% positive cases, corresponding to a negative-to-positive ratio close to 19:1; this motivates imbalance-aware evaluation metrics and training strategies.
3. `bmi` is the only column with explicit numeric missingness (around 3.9% of rows), while `smoking_status` encodes informative missingness through an `Unknown` level that is both large and non-randomly distributed.
4. Age is the single strongest unconditional correlate of stroke, with stroke prevalence increasing sharply after approximately 60 years of age as visible in the per-decile rates and the age-glucose scatter plot.
5. `avg_glucose_level` is right-skewed and visibly bimodal, consistent with a diabetic subpopulation; stroke cases concentrate in the upper-glucose cluster, indicating that glucose carries discriminative information beyond age alone.
6. `bmi` is approximately log-normal with a long upper tail; the IQR rule flags a non-trivial fraction of upper-tail BMI values as outliers but these are clinically plausible and should not be blindly removed.
7. Categorical attributes `hypertension`, `heart_disease`, and `ever_married` show the largest stroke-rate differentials across levels, and chi-squared tests confirm statistically significant association with the target.
8. `gender` and `Residence_type` show only marginal differences in stroke prevalence across their levels, and their association with the target is weak in both chi-squared and Spearman terms; the dataset also contains a single `Other` gender record that must be handled explicitly.
9. `work_type` carries a confound with age — the `children` and `Never_worked` categories have essentially zero stroke prevalence because they are composed of younger patients — so its apparent effect must be interpreted in a multivariate rather than marginal sense.
10. Spearman correlations between continuous features are modest, with no pair exceeding a magnitude that would raise immediate multicollinearity concerns, suggesting that the three numeric predictors can be retained jointly in downstream models.